# ELMFIRE and Earth2 proof of concept

This notebook mirrors the local-first workflow in `fire_poc`.


## 1. setup

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path.cwd().resolve().parents[0]
SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fire_poc.compare import configure_logging
from fire_poc.config import DomainConfig
from fire_poc.earth2_provider import Earth2StudioGEFSProvider
from fire_poc.elmfire_runner import ElmfireRunner, build_proxy_benchmark
from fire_poc.mock_provider import MockEarth2Provider
from fire_poc.plotting import create_comparison_figure
from fire_poc.regime_model import RegimeAwareModel

configure_logging()
domain = DomainConfig()
workdir = PROJECT_ROOT / "outputs" / "notebook_demo"
workdir.mkdir(parents=True, exist_ok=True)
print(workdir)


## 2. choose provider

In [ ]:
provider_name = "mock"  # switch to "earth2" after installing earth2studio
provider = MockEarth2Provider() if provider_name == "mock" else Earth2StudioGEFSProvider()
provider


## 3. fetch forcing

In [ ]:
forcing = provider.fetch_forcing(domain=domain, duration_hours=12.0, step_hours=1.0)
(workdir / "forcing.json").write_text(json.dumps(forcing.to_serializable(), indent=2))
forcing.weather_summary()


## 4. write ELMFIRE case

In [ ]:
runner = ElmfireRunner(executable_command=None)
case_status = runner.run(case_dir=workdir / "case", domain=domain, forcing=forcing)
case_status.message


## 5. run benchmark if available

In [ ]:
benchmark_result = None if case_status.run_completed else build_proxy_benchmark(forcing=forcing, domain=domain)
case_status.benchmark_type


## 6. run regime-aware model

In [ ]:
regime_result = RegimeAwareModel().run(forcing=forcing, domain=domain)
regime_result.metadata


## 7. make figure

In [ ]:
figure_path = create_comparison_figure(
    forcing=forcing,
    benchmark_status=case_status,
    regime_result=regime_result,
    benchmark_result=benchmark_result,
    output_path=workdir / "figures" / "elmfire_vs_regime_model.png",
    title="ELMFIRE benchmark scaffold vs regime-aware fire model",
)
figure_path


## 8. inspect generated files

In [ ]:
sorted(str(path.relative_to(workdir)) for path in workdir.rglob("*") if path.is_file())
